## Exploring the dataset

In [1]:
import pandas as pd

In [7]:
BIG5_COLS = ["agreeableness", "openness", "conscientiousness",
             "extraversion", "neuroticism"]


def filter_comments(profiles_path, comments_path, output_path,
                     english_only=True, chunksize=200_000):
    # 1. Les brukerprofilene (liten fil, trygt å lese i ett steg)
    profiles = pd.read_csv(profiles_path)
    print(f"Fant {len(profiles)} brukerprofiler totalt.")

    # Behold kun brukere som har alle fem Big Five-skårer utfylt
    has_all_traits = profiles[BIG5_COLS].notna().all(axis=1)
    big5_profiles = profiles.loc[has_all_traits, ["author"] + BIG5_COLS].copy()
    big5_users = set(big5_profiles["author"])
    print(f"{len(big5_users)} brukere har alle fem Big Five-skårer.")

    # Lagre denne lille, rensede profiltabellen for seg selv også –
    # du trenger den senere for å koble på fasit-verdiene (y)
    big5_profiles.to_csv("../data/big5_profiles_filtered.csv", index=False)
    print("Lagret big5_profiles_filtered.csv (bare disse brukerne + skårene)")

    # 2. Les den store kommentarfilen i biter (chunks), behold bare
    #    rader fra de 1608 brukerne, skriv fortløpende til en ny fil
    first_chunk = True
    total_kept = 0
    total_seen = 0

    keep_cols = ["author", "subreddit", "body", "created_utc",
                 "word_count", "lang"]

    for i, chunk in enumerate(pd.read_csv(comments_path, chunksize=chunksize,
                                           low_memory=False)):
        total_seen += len(chunk)

        mask = chunk["author"].isin(big5_users)
        if english_only and "lang" in chunk.columns:
            mask &= (chunk["lang"] == "en")

        filtered = chunk.loc[mask, [c for c in keep_cols if c in chunk.columns]]

        filtered.to_csv(output_path, mode="w" if first_chunk else "a",
                         header=first_chunk, index=False)
        first_chunk = False
        total_kept += len(filtered)

        if i % 5 == 0:
            print(f"  ...lest {total_seen:,} rader totalt, "
                  f"beholdt {total_kept:,} hittil")

    print(f"\nFerdig. {total_kept:,} av {total_seen:,} kommentarer "
          f"skrevet til {output_path}")


if __name__ == "__main__":
    filter_comments(
        profiles_path="../data/author_profiles.csv",
        comments_path="../data/all_comments_since_2015.csv",
        output_path="../data/filtered_comments.csv",
        english_only=True,
    )

Fant 10295 brukerprofiler totalt.
1568 brukere har alle fem Big Five-skårer.
Lagret big5_profiles_filtered.csv (bare disse brukerne + skårene)
  ...lest 200,000 rader totalt, beholdt 6,514 hittil
  ...lest 1,200,000 rader totalt, beholdt 40,050 hittil
  ...lest 2,200,000 rader totalt, beholdt 74,488 hittil
  ...lest 3,200,000 rader totalt, beholdt 107,269 hittil
  ...lest 4,200,000 rader totalt, beholdt 141,839 hittil
  ...lest 5,200,000 rader totalt, beholdt 176,217 hittil
  ...lest 6,200,000 rader totalt, beholdt 211,068 hittil
  ...lest 7,200,000 rader totalt, beholdt 245,878 hittil
  ...lest 8,200,000 rader totalt, beholdt 279,651 hittil
  ...lest 9,200,000 rader totalt, beholdt 315,306 hittil
  ...lest 10,200,000 rader totalt, beholdt 402,383 hittil
  ...lest 11,200,000 rader totalt, beholdt 1,272,383 hittil
  ...lest 12,200,000 rader totalt, beholdt 2,142,348 hittil
  ...lest 13,200,000 rader totalt, beholdt 2,610,760 hittil
  ...lest 14,200,000 rader totalt, beholdt 2,661,872 hi

## **Explore**

Utforsk de filtrerte PANDORA-dataene.

In [12]:
# Subreddits som direkte avslører personlighetstype -- disse bør IKKE
# brukes som nettverksfeature, siden de ville "lekke" fasiten (label
# leakage). Utvid listen hvis du finner flere slike i dine data.
PERSONALITY_SUBREDDITS = {
    "mbti", "intp", "infj", "infp", "intj", "entp", "enfp", "enfj", "entj",
    "istj", "isfj", "istp", "isfp", "estj", "esfj", "estp", "esfp",
    "enneagram", "myersbriggs", "personalitycafe", "MBTIPlus",
}


def explore(comments_path="../data/filtered_comments.csv",
            profiles_path="../data/big5_profiles_filtered.csv"):

    comments = pd.read_csv(comments_path)
    profiles = pd.read_csv(profiles_path)

    print("=" * 60)
    print("GRUNNTALL")
    print("=" * 60)
    print(f"Antall kommentarer: {len(comments):,}")
    print(f"Antall brukere med Big Five-skårer: {len(profiles)}")
    print(f"Antall brukere som faktisk har kommentarer i dataen: "
          f"{comments['author'].nunique()}")

    print("\n" + "=" * 60)
    print("KOMMENTARER PER BRUKER")
    print("=" * 60)
    per_user = comments.groupby("author").size()
    print(per_user.describe())
    print(f"\nBrukere med færre enn 20 kommentarer: "
          f"{(per_user < 20).sum()} ({(per_user < 20).mean():.1%})")

    print("\n" + "=" * 60)
    print("LENGDE PÅ KOMMENTARER (word_count)")
    print("=" * 60)
    if "word_count" in comments.columns:
        print(comments["word_count"].describe())
        print(f"Kommentarer med færre enn 5 ord: "
              f"{(comments['word_count'] < 5).sum():,}")

    print("\n" + "=" * 60)
    print("SUBREDDITS")
    print("=" * 60)
    print(f"Antall unike subreddits totalt: {comments['subreddit'].nunique()}")
    sub_counts = comments["subreddit"].value_counts()
    print("\nTopp 15 mest aktive subreddits:")
    print(sub_counts.head(15))

    found_personality_subs = set(comments["subreddit"].str.lower()) & PERSONALITY_SUBREDDITS
    n_leaky_comments = comments["subreddit"].str.lower().isin(PERSONALITY_SUBREDDITS).sum()
    print(f"\n⚠️  Personlighets-relaterte subreddits funnet i dataene: "
          f"{sorted(found_personality_subs)}")
    print(f"⚠️  Antall kommentarer i disse (bør trolig filtreres bort fra "
          f"nettverksfeatureet): {n_leaky_comments:,} "
          f"({n_leaky_comments / len(comments):.1%} av alle kommentarer)")

    print("\n" + "=" * 60)
    print("MANGLENDE VERDIER OG DUPLIKATER")
    print("=" * 60)
    print("Manglende verdier per kolonne (comments):")
    print(comments.isna().sum())
    n_deleted = comments["body"].isin(["[deleted]", "[removed]"]).sum()
    print(f"\nSlettede/fjernede kommentarer: {n_deleted:,}")
    n_dupes = comments.duplicated(subset=["author", "body"]).sum()
    print(f"Duplikate (bruker, tekst)-par: {n_dupes:,}")

    print("\n" + "=" * 60)
    print("BIG FIVE-SKÅRER")
    print("=" * 60)
    big5_cols = ["agreeableness", "openness", "conscientiousness",
                 "extraversion", "neuroticism"]
    print(profiles[big5_cols].describe())

    print("\n" + "=" * 60)
    print("TIDSSPENN")
    print("=" * 60)
    if "created_utc" in comments.columns:
        dates = pd.to_datetime(comments["created_utc"], unit="s")
        print(f"Eldste kommentar: {dates.min()}")
        print(f"Nyeste kommentar: {dates.max()}")


if __name__ == "__main__":
    explore()

GRUNNTALL
Antall kommentarer: 2,830,311
Antall brukere med Big Five-skårer: 1568
Antall brukere som faktisk har kommentarer i dataen: 1568

KOMMENTARER PER BRUKER
count     1568.000000
mean      1805.045281
std       4025.093031
min          1.000000
25%         70.000000
50%        374.500000
75%       1682.500000
max      52066.000000
dtype: float64

Brukere med færre enn 20 kommentarer: 209 (13.3%)

LENGDE PÅ KOMMENTARER (word_count)
count    2.830311e+06
mean     4.340285e+01
std      7.827686e+01
min      1.000000e+00
25%      9.000000e+00
50%      2.000000e+01
75%      4.600000e+01
max      3.000000e+03
Name: word_count, dtype: float64
Kommentarer med færre enn 5 ord: 276,832

SUBREDDITS
Antall unike subreddits totalt: 15391

Topp 15 mest aktive subreddits:
subreddit
AskReddit                170378
OkCupid                  158656
popheads                  81736
mbti                      75967
entp                      72067
INTP                      50177
politics                

## Clean dataset

Tar inn filtered_comments.csv + big5_profiles_filtered.csv og gjør følgende:

1. Fjerner duplikate (bruker, tekst)-par
2. Fjerner [deleted]/[removed]-kommentarer
3. Fjerner brukere med færre enn MIN_COMMENTS kommentarer
4. Lager en egen "clean" kommentar-tabell for tekstmodellen
5. Lager en egen subreddit-tabell for nettverksfeatureet, der personlighets-relaterte subreddits er fjernet (for å unngå
   label leakage), men lagrer også en versjon MED dem i tilfelle du vil vise fram forskjellen i rapporten



In [10]:
MIN_COMMENTS_PER_USER = 20

# Subreddits som direkte avslører personlighetstype -- fjernes fra
# nettverksfeatureet for å unngå label leakage (bekreftet i dataene:
# ~10% av alle kommentarer ligger i disse)
PERSONALITY_SUBREDDITS = {
    "mbti", "intp", "infj", "infp", "intj", "entp", "enfp", "enfj", "entj",
    "istj", "isfj", "istp", "isfp", "estj", "esfj", "estp", "esfp",
    "enneagram", "myersbriggs", "personalitycafe", "MBTIPlus",
}



def clean(comments_path="../data/filtered_comments.csv",
          profiles_path="../data/big5_profiles_filtered.csv"):

    comments = pd.read_csv(comments_path)
    profiles = pd.read_csv(profiles_path)
    print(f"Start: {len(comments):,} kommentarer, {len(profiles)} brukere")

    # 1. Fjern slettede/fjernede kommentarer
    comments = comments[~comments["body"].isin(["[deleted]", "[removed]"])]
    print(f"Etter å fjerne slettede/fjernede: {len(comments):,} kommentarer")

    # 2. Fjern duplikate (bruker, tekst)-par (behold første forekomst)
    before = len(comments)
    comments = comments.drop_duplicates(subset=["author", "body"])
    print(f"Fjernet {before - len(comments):,} duplikater, "
          f"{len(comments):,} kommentarer igjen")

    # 3. Fjern brukere med for få kommentarer
    counts = comments.groupby("author").size()
    keep_users = counts[counts >= MIN_COMMENTS_PER_USER].index
    comments = comments[comments["author"].isin(keep_users)]
    profiles = profiles[profiles["author"].isin(keep_users)]
    print(f"Brukere med minst {MIN_COMMENTS_PER_USER} kommentarer: "
          f"{len(keep_users)}")
    print(f"Kommentarer igjen: {len(comments):,}")

    # 4. Lagre den rensede tekst-tabellen (brukes til tekstmodellen)
    comments.to_csv("../data/comments_clean.csv", index=False)
    profiles.to_csv("../data/profiles_clean.csv", index=False)
    print("\nLagret comments_clean.csv og profiles_clean.csv "
          "(til tekstmodellen)")

    # 5. Lag nettverkstabellen: subreddit-deltakelse PER BRUKER,
    #    med personlighets-subreddits fjernet
    is_leaky = comments["subreddit"].str.lower().isin(PERSONALITY_SUBREDDITS)
    network_comments = comments[~is_leaky]

    subreddit_counts = (
        network_comments
        .groupby(["author", "subreddit"])
        .size()
        .reset_index(name="count")
    )
    subreddit_counts.to_csv("../data/subreddit_counts_clean.csv", index=False)
    print(f"\nLagret subreddit_counts_clean.csv "
          f"({is_leaky.sum():,} kommentarer i personlighets-subreddits "
          f"ble utelatt fra nettverksfeatureet)")
    print(f"Unike subreddits igjen i nettverksfeatureet: "
          f"{network_comments['subreddit'].nunique()}")

    print("\nFerdig! Du kan nå bygge:")
    print("  - tekstmodellen fra comments_clean.csv + profiles_clean.csv")
    print("  - subreddit-vektoren fra subreddit_counts_clean.csv")


if __name__ == "__main__":
    clean()

Start: 2,830,311 kommentarer, 1568 brukere
Etter å fjerne slettede/fjernede: 2,830,274 kommentarer
Fjernet 242,699 duplikater, 2,587,575 kommentarer igjen
Brukere med minst 20 kommentarer: 1356
Kommentarer igjen: 2,585,996

Lagret comments_clean.csv og profiles_clean.csv (til tekstmodellen)

Lagret subreddit_counts_clean.csv (233,224 kommentarer i personlighets-subreddits ble utelatt fra nettverksfeatureet)
Unike subreddits igjen i nettverksfeatureet: 15254

Ferdig! Du kan nå bygge:
  - tekstmodellen fra comments_clean.csv + profiles_clean.csv
  - subreddit-vektoren fra subreddit_counts_clean.csv
